# PodCut AI on a free Kaggle GPU

Runs the same pipeline as the web app, end to end, on Kaggle.

1. **Settings → Accelerator: GPU T4**, **Internet: On** (needs a phone-verified account)
2. **Add-ons → Secrets**: add `GROQ_API_KEY` (or `XAI_API_KEY`)
3. Either attach a dataset with your video, or put a link in `SOURCE` below
4. **Run all**

In [ ]:
!git clone -q https://github.com/Nimesh000/podcut-ai.git /kaggle/working/podcut-ai || (cd /kaggle/working/podcut-ai && git pull -q)
%cd /kaggle/working/podcut-ai
!pip install -q -r requirements.txt
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)

In [ ]:
import os, glob
# make the pip-installed CUDA libraries visible to faster-whisper (CTranslate2)
try:
    import nvidia.cublas.lib, nvidia.cudnn.lib
    libs = [os.path.dirname(nvidia.cublas.lib.__file__), os.path.dirname(nvidia.cudnn.lib.__file__)]
    os.environ['LD_LIBRARY_PATH'] = ':'.join(libs + [os.environ.get('LD_LIBRARY_PATH', '')])
except ImportError:
    pass

# API key from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    for name in ('GROQ_API_KEY', 'XAI_API_KEY'):
        try:
            os.environ[name] = sc.get_secret(name); print('Loaded secret', name)
        except Exception:
            pass
except ImportError:
    print('Not on Kaggle - set GROQ_API_KEY yourself')

In [ ]:
# A link (YouTube / Google Drive / direct .mp4) OR leave empty to use the first video in an attached dataset
SOURCE = ''
TITLE, SHOW, GUEST, ROLE = '', '', '', ''
SECONDS = 150

if not SOURCE:
    files = sorted(glob.glob('/kaggle/input/**/*.mp4', recursive=True) + glob.glob('/kaggle/input/**/*.mp3', recursive=True))
    assert files, 'Attach a dataset with a video or set SOURCE'
    SOURCE = files[0]
print('Input:', SOURCE)

In [ ]:
import shlex, subprocess
cmd = ['python', '-m', 'podcut', 'run', SOURCE, '--seconds', str(SECONDS)]
for flag, val in (('--title', TITLE), ('--show', SHOW), ('--guest', GUEST), ('--role', ROLE)):
    if val: cmd += [flag, val]
print(' '.join(map(shlex.quote, cmd)))
!{' '.join(map(shlex.quote, cmd))} 2>&1 | grep -v '\$ ffmpeg'

In [ ]:
import json, shutil
from IPython.display import Video, Markdown, display
job = sorted(glob.glob('jobs/*'))[-1]
out = os.path.join(job, 'output')
shutil.copytree(out, '/kaggle/working/output', dirs_exist_ok=True)
print(open(os.path.join(job, 'pipeline.log')).read()[-1500:])
display(Markdown(open(os.path.join(out, 'edit_report.md')).read()))
display(Video('/kaggle/working/output/final_video.mp4', embed=False, width=720))